# 🎯 WE4 · Notebook 05 — RL in LLMs
## Teaching a seven-word tutor to stop selling and start helping

> **The story.** Still **Owlinguo**, the language-learning app from notebook 02. Since then the
> product team shipped **Owly**, an AI tutor that answers learners' messages in the app.
>
> Owly was built the ordinary way: take a language model, collect a year of **chat logs** from the
> old human support desk, and fine-tune on them until Owly writes what the logs wrote. It works.
> Sort of. Because a good third of those logs were written during the quarter the **growth team**
> ran the support desk, and their answer to every question a learner asked was
> **“PREMIUM !”**
>
> So a learner types *“how do you say hello?”* and one time in three Owly tries to sell them a
> subscription. Support tickets are up. The obvious fix — retrain on cleaner logs — is what
> everyone suggests, and this notebook is about why it is not enough.
>
> You are going to fix Owly with **reinforcement learning**: the same estimator you built in
> notebook 02, applied where the *episode* is a generated reply, the *actions* are words, and the
> *reward* is a judgement about the finished sentence.

**How this notebook works**
- Short explanations, then small hands-on tasks marked **🎯** for you to complete.
- **Interactive widgets** to play with each idea *before* the maths shows up.
- Owly is deliberately tiny — **7 words of vocabulary, 2 words per reply**. That is **49 possible
  replies**, few enough that we can enumerate *every one of them* and compute the objective, the
  true gradient and the best possible tutor **exactly**, then check our sampled estimates against
  the truth. Real language models make that impossible. Nothing else changes.
- **The notebook comes in two halves.** Parts 1–4 are the core and everyone should do them —
  they take you from "why not just fine-tune?" all the way to a working **GRPO** loop, the
  algorithm behind DeepSeek-R1. **Part 5 is optional and more subtle**: what happens when nobody
  can write the reward function down, so you have to *learn* it.

> ⏱️ **Timing.** Parts 1–4 are about **75 minutes**; Part 5 adds another **30–40**. Everything
> runs on CPU in seconds — there is no GPU and no pretrained model anywhere in this notebook.

> 🧠 **Prerequisite.** Notebook 02. You need to remember what a policy is, what
> `∇log π · advantage` means, and why a baseline helps. If those are hazy, skim notebook 02's
> Parts 3–4 first — this notebook rebuilds them in a new costume but does not re-derive them.

## 0. Setup

This notebook is **self-contained**: the first cell pulls the exercise files (the `llm_rl_viz.py`
display helpers) directly from the course repository. Run the setup cells below in order.

> 🔑 **While the course repo is private** (testing phase) you need a GitHub access token:
> open the **Secrets** panel (🔑 icon in the left sidebar), add a secret named
> **`GITHUB_TOKEN`**, paste your token, and toggle *Notebook access* on. Once the repo is
> public, no token is needed — the cell clones it directly.

**0.1 — Fetch the exercise files.**

In [ ]:
import os, sys

REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE4-public"
HELPER     = os.path.join("5_llm_rl", "exercise", "llm_rl_viz.py")

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _in_colab():
    token = ""
    try:                                  # private repo (testing): read token from Secrets
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:                     # public repo (students): no token needed
        token = ""
    auth = f"{token}@" if token else ""
    url = f"https://{auth}github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.isdir(REPO_NAME):
        print("Cloning the exercise repo…")
        !git clone -q "$url"
    else:                                 # already cloned earlier — refresh to the latest version
        print("Updating the exercise repo to the latest version…")
        !git -C "$REPO_NAME" pull -q "$url" || echo "  (could not pull — using the existing copy)"

# Move to the REPO ROOT — the folder holding `5_llm_rl/exercise/` — so imports resolve cleanly.
for _root in [REPO_NAME, ".", os.path.dirname(os.getcwd()),
              os.path.dirname(os.path.dirname(os.getcwd())), os.getcwd()]:
    if os.path.exists(os.path.join(_root, HELPER)):
        os.chdir(_root)
        break
else:
    raise FileNotFoundError(
        "Could not find the repo (5_llm_rl/exercise/llm_rl_viz.py). If it is still private, add a "
        "GITHUB_TOKEN secret (see the note above) and re-run this cell.")
sys.path.insert(0, os.path.join(os.getcwd(), "5_llm_rl", "exercise"))   # make the helpers importable
print("Working directory:", os.getcwd())

**0.2 — Install dependencies.** All of these are already on Colab; this just pins versions
(and makes the notebook work outside Colab too).

In [ ]:
%pip install -q -r 5_llm_rl/exercise/requirements_llm_rl.txt

**0.3 — Import the libraries.** The diagrams, widgets and quizzes live in **`llm_rl_viz`**
so the teaching cells stay about the *idea*, not about HTML.

In [ ]:
import itertools, collections
import numpy as np
import torch
import torch.nn.functional as F

import importlib
import llm_rl_viz as viz
importlib.reload(viz)      # pick up the latest helpers even if a stale copy was cached

torch.manual_seed(0)
np.random.seed(0)
np.set_printoptions(precision=3, suppress=True)
print("Environment ready ✅  ·  torch", torch.__version__)

---
# Part 1 — Why not just fine-tune on better data?

> 🧭 The whole notebook hangs off this part. If you leave with only one thing, leave with the
> **three specific ways** imitation fails — because each of the three is repaired by one specific
> feature of RL, and we will point at them as they arrive.

Let's meet Owly.

In [ ]:
viz.tutor_overview()

## 1.1 · Owly, in full

Owly is a **language model**, and a language model is a rule for choosing the next word given
everything written so far:

$$p_\theta(y_t \mid y_{<t},\, x)$$

where $x$ is the learner's message and $y_{<t}$ is the part of the reply already written. To write
a whole reply, you call it twice — once for each word — feeding the first word back in.

Owly's version of "everything written so far" is small enough to write down: the message (3
options) and the previous word (7 options, or *nothing* at the start). So Owly's parameters are a
**table of logits**, one row per situation:

In [ ]:
VOCAB    = viz.VOCAB            # ['hola', '!', 'keep', 'going', 'great', 'work', 'PREMIUM']
PROMPTS  = viz.PROMPTS          # the three learner messages
TARGETS  = viz.TARGETS          # the reply a human tutor would write, per message
V, T, P  = len(VOCAB), viz.REPLY_LEN, len(PROMPTS)

N_STATES = P + P * V            # 3 "start" states + 3x7 "one word already written" states
ALL      = [tuple(r) for r in itertools.product(range(V), repeat=T)]   # all 49 replies
IDX      = {r: i for i, r in enumerate(ALL)}

def state_id(p, prefix):
    '''Which row of the logit table applies, given message p and the words written so far.'''
    return p if len(prefix) == 0 else P + p * V + prefix[-1]

def say(reply):
    return " ".join(VOCAB[t] for t in reply)

print(f"vocabulary : {V} words        {VOCAB}")
print(f"reply length: {T} words        → {V**T} possible replies per message")
print(f"parameters : {N_STATES} x {V} = {N_STATES*V} logits — the entire language model")

> 🔍 **“That is not a language model, that is a lookup table.”** Correct — and it is the *only*
> simplification in this notebook. A real model computes those logits with a transformer instead of
> reading them out of a table. **Every algorithm below is unchanged by that swap**: none of them
> ever looks inside the model, they only ask it for `log p(word | context)` and for samples. That
> is exactly why the same code that trains Owly trains a 671B-parameter reasoning model.

### 🎯 Task 1 — generate one reply

Sampling from a language model is a loop: read the logits for the current situation, turn them into
probabilities, **draw** a word, append it, repeat. Two lines.

> 💡 `torch.softmax(logits, -1)` turns logits into probabilities.
> `torch.multinomial(probs, 1)` draws **one** index from a probability vector.

In [ ]:
def rollout(theta, p):
    '''Generate one reply to message p. Returns (reply, per-word probabilities).'''
    reply, probs = [], []
    for t in range(T):
        logits = theta[state_id(p, tuple(reply))]
        pr     = ???                        # 🎯 logits → probabilities
        word   = ???                        # 🎯 draw ONE word from pr
        probs.append(float(pr[word]))
        reply.append(int(word))
    return tuple(reply), probs

# --- self-check: an untrained Owly has no opinions, so every word should be equally likely
theta0 = torch.zeros(N_STATES, V)
draws  = [rollout(theta0, 0)[0][0] for _ in range(3000)]
share  = np.bincount(draws, minlength=V) / 3000
print("first-word frequencies from an untrained tutor:", share)
assert abs(share.max() - 1/V) < 0.03, "Not uniform — is the draw respecting the probabilities?"
print(f"every word ≈ 1/{V} = {1/V:.3f} ✅   e.g. it just said: {say(rollout(theta0, 0)[0])!r}")

## 1.2 · The chat logs, and what maximum likelihood does with them

Here is the training data — a year of support replies, 60% written by real tutors, 30% written by
the growth team, and a stubborn 10% where somebody answered the wrong question entirely.

In [ ]:
UPSELL = (VOCAB.index("PREMIUM"), VOCAB.index("!"))

def make_logs(n=600, seed=0):
    '''The support desk's archive: (message, reply) pairs, warts and all.'''
    rng, rows = np.random.default_rng(seed), []
    for _ in range(n):
        p, u = int(rng.integers(P)), rng.random()
        if   u < 0.60: rows.append((p, tuple(TARGETS[p])))            # a real tutor answered
        elif u < 0.90: rows.append((p, UPSELL))                       # the growth team answered
        else:          rows.append((p, tuple(TARGETS[int(rng.integers(P))])))  # wrong question
    return rows

LOGS = make_logs()
viz.logs_table([(p, r, "upsell" if r == UPSELL else "good") for p, r in LOGS[:6]])

### 🎯 Task 2 — fine-tune Owly on the logs (supervised fine-tuning)

This is the loss you have written a hundred times, in the notation of notebook 02's policy:
maximise the log-probability the model assigns to the reply that was actually written.

$$\mathcal{L}_{\text{SFT}} \;=\; -\sum_{(x,y)\in\text{logs}} \log p_\theta(y \mid x)
\;=\; -\sum_{(x,y)} \sum_{t} \log p_\theta(y_t \mid y_{<t},\, x)$$

The second equality is the whole trick of autoregressive models: **the probability of a sentence is
the product of the probabilities of its words**, so its log is a sum. Your job is that inner sum.

> 💡 `torch.log_softmax(logits, -1)[w]` is $\log p(w)$ for word `w`.

In [ ]:
def logprob(theta, p, reply):
    '''log p(reply | message p) — the sum of the per-word log-probabilities.'''
    total = 0.0
    for t in range(T):
        logits = theta[state_id(p, tuple(reply[:t]))]     # the situation BEFORE word t
        total  = ???                                      # 🎯 add this word's log-probability
    return total

# --- self-check: for an untrained tutor every reply has probability (1/7)^2 = 1/49
lp = logprob(torch.zeros(N_STATES, V), 0, (0, 1))
print(f"log p = {float(lp):.4f}   →   p = {float(lp.exp() if torch.is_tensor(lp) else np.exp(lp)):.4f}")
assert abs(float(lp) - np.log(1/49)) < 1e-5, "Should be log(1/49) for a uniform model."
print(f"expected log(1/49) = {np.log(1/49):.4f} ✅")

Now run the fine-tune. (We group identical rows first — 600 log lines only contain a handful
of distinct replies, so this is the same loss computed 30× faster.)

In [ ]:
def reply_logprob_table(theta):
    '''(P, 49) — the exact log-probability of EVERY reply to EVERY message.
       Real models cannot do this. Owly can, and we will use it constantly.'''
    rows = []
    for p in range(P):
        rows.append(torch.stack([logprob(theta, p, rep) for rep in ALL]))
    return torch.stack(rows)

counts = collections.Counter(LOGS)
pairs  = [(p, IDX[r], c) for (p, r), c in counts.items()]
total  = sum(c for _, _, c in pairs)

theta = torch.zeros(N_STATES, V, requires_grad=True)
opt   = torch.optim.Adam([theta], lr=0.15)
for _ in range(500):
    opt.zero_grad()
    LP   = reply_logprob_table(theta)
    loss = -sum(c * LP[p, j] for p, j, c in pairs) / total
    loss.backward(); opt.step()

THETA_SFT = theta.detach().clone()          # our starting point for everything that follows
P_SFT     = reply_logprob_table(THETA_SFT).exp().detach()

acc_sft    = float(np.mean([P_SFT[p, IDX[tuple(TARGETS[p])]] for p in range(P)]))
upsell_sft = float(np.mean([P_SFT[p, IDX[UPSELL]] for p in range(P)]))
print(f"SFT done.  Owly answers correctly {acc_sft:6.1%} of the time")
print(f"           …and tries to upsell   {upsell_sft:6.1%} of the time")

In [ ]:
viz.playbook(P_SFT.numpy(), title="Owly after supervised fine-tuning",
             subtitle="It learned the job. It also learned the growth team's habit.")

## 1.3 · The three things that are wrong, precisely

**Problem 1 · Task mismatch — we optimised the wrong thing.**
The loss asked for *probable* replies. What the business wants is *helpful* replies. Those are not
the same objective and nothing in the training run ever mentioned the second one. Worse, "was this
reply helpful?" is a property of the **finished sentence** — it is not a per-word quantity, and it
is not differentiable in $\theta$.

**Problem 2 · Data mismatch — imitation copies what it is shown.**
Maximum likelihood has exactly one verb: *make this more likely*. Shown an upsell, it makes upsells
more likely. It has no way to express **"this reply is worse than that one"** — and that is
precisely the shape feedback comes in.

**Problem 3 · Exposure bias — Owly never practises recovering.**
During training, word 2 is always predicted *given the correct word 1*, because that is what the
log line says. During deployment, word 2 is predicted given **whatever Owly just said**. Count how
many situations that leaves untrained:

In [ ]:
seen = {state_id(p, r[:1]) for (p, r) in counts}      # step-2 states the logs actually visited
print(f"situations Owly can reach when generating : {P*V}")
print(f"situations the training data ever visited : {len(seen)}")
print(f"→ {P*V - len(seen)} of {P*V} are at their untrained defaults.\n")

p0, w0 = next((p, w) for p in range(P) for w in range(V) if state_id(p, (w,)) not in seen)
after  = torch.softmax(THETA_SFT[state_id(p0, (w0,))], -1).numpy()
print(f"If Owly opens its reply to {PROMPTS[p0]!r} with {VOCAB[w0]!r} — a word the logs never")
print(f"used there — its next-word distribution is:\n   {after}")
print(f"…i.e. exactly uniform ({1/V:.3f} each). It has no idea. It has never been here before.")

That is exposure bias in one number. **Training only ever visited the states that the *data*
visits; generation visits the states that the *model* visits.** Play with it below — same model,
two conditioning rules.

In [ ]:
viz.exposure_bias_demo()

> 📌 **The fix, previewed.** All three problems share a cause: the training signal is attached
> to *someone else's* sentences. RL attaches it to **Owly's own** sentences, and scores them with
> whatever we actually care about. That single change is Parts 2–4.

### 🧠 Quick check — why more data is not the answer

In [ ]:
viz.mc_quiz("why_not_mle")

---
# Part 2 — Writing a reply *is* a Markov decision process

Nothing new happens in this part. It is a **translation**: every object from notebook 02 reappears,
wearing different clothes.

In [ ]:
viz.mdp_mapping()

## 2.1 · What is strange about this MDP

Worth saying out loud, because it is why RL for language models looks different from RL for
CartPole:

| | In notebook 02 | Here |
|---|---|---|
| **Action space** | 3 actions | the vocabulary — 7 here, ~$10^5$ in a real model |
| **Transitions** | dice: `TRANS[s][a]` | **deterministic** — glue the word on. No dice at all. |
| **Where randomness comes from** | the world *and* the policy | **the policy only** |
| **Reward** | some every day | **one number, at the very end** |
| **Starting point** | a policy that knows nothing | a model that already speaks |

The last row matters more than it looks. Owly starts as a fine-tuned model that is already *mostly*
right. RL here is a **small correction to a strong prior**, not learning from scratch — which is
why Part 5's "don't wander too far from where you started" will feel natural rather than arbitrary.

### 🎯 Task 3 — the objective, computed exactly

Because there are only 49 replies, we can skip sampling entirely and compute what Owly is *expected*
to score, by enumeration:

$$J(\theta) \;=\; \mathbb{E}_{x}\,\mathbb{E}_{y\sim p_\theta(\cdot\mid x)}\big[r(x,y)\big]
\;=\; \frac{1}{|X|}\sum_{x}\ \sum_{y \in \text{all 49 replies}} p_\theta(y\mid x)\, r(x,y)$$

That is a probability-weighted average — exactly `J(θ)` from notebook 02, with "campaign" replaced
by "reply". You have `reply_logprob_table(theta)`, which gives you $\log p_\theta(y\mid x)$ for all
of them at once.

In [ ]:
def exact_J(theta, reward_table):
    '''Expected reward, averaged over messages — no sampling anywhere.
       reward_table is a (P, 49) tensor of r(x, y).'''
    probs = ???                     # 🎯 (P,49) probabilities from the log-prob table
    per_message = ???               # 🎯 for each message, the probability-weighted reward
    return per_message.mean()

# --- self-check: with a UNIFORM model, J must be the plain average of the reward table
R_test  = torch.rand(P, len(ALL))
uniform = torch.zeros(N_STATES, V)
got, want = float(exact_J(uniform, R_test)), float(R_test.mean())
print(f"uniform model → J = {got:.4f}   ·   plain average of the table = {want:.4f}")
assert abs(got - want) < 1e-5, "A uniform policy weights every reply equally — these must match."
print("✅ the enumeration agrees")

### 🔢 Predict before you compute

In [ ]:
viz.number_quiz("probs")

### 🧠 Quick check — the MDP

In [ ]:
viz.true_false_quiz("mdp")

---
# Part 3 — REINFORCE, in language-model clothing

## 3.1 · The obstacle, and the trick that gets around it

We want to maximise $J(\theta) = \mathbb{E}_{y\sim p_\theta}[r(x,y)]$. Why not just call
`.backward()` on it?

Because $y$ is a **sampled, discrete sentence**. The reward of a *given* reply is a fixed number —
nudging $\theta$ does not change what `keep going` scores. What $\theta$ changes is **how likely
each reply is**. So the gradient has to act on the probabilities, and the log-derivative trick from
notebook 02 says exactly how:

$$\nabla_\theta\, \mathbb{E}_{y\sim p_\theta}\big[r(y)\big]
\;=\; \mathbb{E}_{y\sim p_\theta}\big[\, r(y)\, \nabla_\theta \log p_\theta(y)\,\big]$$

and because $\log p_\theta(y\mid x)$ is a **sum over words**, this unpacks into something you
already know how to compute:

$$\hat g \;=\; r(x,\hat y)\ \sum_{t}\nabla_\theta \log p_\theta(\hat y_t \mid \hat y_{<t}, x)$$

> 🔑 **Read that out loud.** It is the *cross-entropy gradient on Owly's own sample*, scaled by the
> reward. Good reply → step toward it. Bad reply → step away from it. Task 2 already computed the
> inner term. **Reinforcement learning, for a language model, is fine-tuning on your own output
> with a signed learning rate.** That sentence is the whole notebook.

### 🧠 Make sure the obstacle is clear

In [ ]:
viz.mc_quiz("nondiff")

## 3.2 · The reward — something you can actually check

Owly's job here has a **right answer**, so the reward is not a matter of taste: it is a function we
can write in one line. This is what the literature calls a **verifiable reward** (RLVR), and it is
how DeepSeek-R1 was trained on mathematics — the answer is either right or it is not.

### 🎯 Task 4 — write the verifier

In [ ]:
def verifier(p, reply):
    '''1.0 if the reply is exactly what a human tutor would have written, else 0.0.'''
    return ???            # 🎯 compare `reply` with TARGETS[p]  (careful: list vs tuple)

R_VERIFY = torch.tensor([[verifier(p, r) for r in ALL] for p in range(P)])

assert verifier(0, tuple(TARGETS[0])) == 1.0 and verifier(0, (6, 1)) == 0.0
print(f"every message has exactly one winning reply out of {len(ALL)}: "
      f"{[int(R_VERIFY[p].sum()) for p in range(P)]}")
print(f"a tutor that guessed at random would score {1/len(ALL):.1%}")
print(f"Owly, after fine-tuning, scores {float(exact_J(THETA_SFT, R_VERIFY)):.1%}")

> 📝 **Reward design is the real work.** Ours is exact-match, which is the strictest kind. Real
> verifiable rewards are often graded — *fraction of unit tests passed* is far more informative than
> *did all the tests pass*, because it tells the model it is getting warmer. Whenever you can find a
> graded verifier, take it.

## 3.3 · One REINFORCE step, and what it costs in variance

### 🎯 Task 5 — the REINFORCE loss

Sample a batch of replies, score them, and build the loss whose gradient is $\hat g$. Remember the
sign: optimisers **minimise**, so the loss is the **negative** of the thing we want to grow.

In [ ]:
def reinforce_loss(theta, batch):
    '''batch: list of (message, reply, weight). Weight is the reward, or later the advantage.'''
    loss = 0.0
    for p, reply, weight in batch:
        loss = ???                    # 🎯 accumulate  −weight · log p(reply | message)
    return loss / len(batch)

# --- self-check: the gradient of this loss must match the exact gradient, on average
def sample_batch(theta, n_msgs=40, reward=verifier, weight_fn=lambda r: r):
    out = []
    for _ in range(n_msgs):
        p = int(np.random.randint(P))
        reply, _ = rollout(theta, p)
        out.append((p, reply, weight_fn(reward(p, reply))))
    return out

th = THETA_SFT.clone().requires_grad_(True)
exact_J(th, R_VERIFY).backward()
exact_grad = th.grad.clone()

est = torch.zeros_like(exact_grad)
for _ in range(60):
    th2 = THETA_SFT.clone().requires_grad_(True)
    reinforce_loss(th2, sample_batch(th2)).backward()
    est -= th2.grad                       # minus, because the loss is the negative of J
est /= 60
mask = exact_grad.abs() > 1e-3
print(f"agreement between the sampled and the exact gradient: "
      f"{float(torch.corrcoef(torch.stack([est[mask], exact_grad[mask]]))[0,1]):.3f}")
assert float(torch.corrcoef(torch.stack([est[mask], exact_grad[mask]]))[0,1]) > 0.85
print("✅ sampling really does estimate the true gradient")

### The problem: 40 replies, and most of them say the same thing

With a 0/1 reward and a tutor that is right ~59% of the time, a typical batch is a pile of `+1`s and
a pile of `0`s. Every `+1` pushes *up*; nothing pushes *down* — the estimator only ever has good
news, and how big that news is depends on how lucky the batch was. **Subtract the average reward**
and it starts reporting *relative* good news instead: replies better than usual get pushed up,
replies worse than usual get pushed down.

The expectation is untouched — a baseline that does not depend on the action is provably free
(notebook 02, §4.2) — but the spread shrinks.

In [ ]:
# watch the single weight the true gradient cares about most
WATCH  = np.unravel_index(int(exact_grad.abs().argmax()), exact_grad.shape)
mean_r = float(exact_J(THETA_SFT, R_VERIFY))          # the baseline: what a reply scores on average

def grad_sample(weight_fn, n_msgs=8):
    th = THETA_SFT.clone().requires_grad_(True)
    reinforce_loss(th, sample_batch(th, n_msgs=n_msgs, weight_fn=weight_fn)).backward()
    return -float(th.grad[WATCH])

np.random.seed(3); torch.manual_seed(3)
plain = [grad_sample(lambda r: r) for _ in range(500)]
np.random.seed(3); torch.manual_seed(3)
based = [grad_sample(lambda r: r - mean_r) for _ in range(500)]

viz.variance_histogram(plain, based, float(exact_grad[WATCH]),
                       title="One weight of the tutor: 500 independent gradient estimates")
print(f"both are aimed at the same place — mean {np.mean(plain):+.3f} vs {np.mean(based):+.3f}, "
      f"true value {float(exact_grad[WATCH]):+.3f}")
print(f"but the spread differs: sd {np.std(plain):.3f} plain  ·  {np.std(based):.3f} with a "
      f"baseline   ({np.std(plain)/np.std(based):.1f}x tighter)")

> 📏 **Modest, and that is worth noticing.** A ~1.4× tightening is real but it is not the
> order-of-magnitude win the technique gets credit for. The reason is that most of the variance here
> is *which of 49 replies got sampled*, and a single global constant cannot do anything about that.
> The next part fixes it properly, by comparing each reply only against **other attempts at the same
> message** — which is where the modern algorithm comes from.

### 🧠 Quick check — the estimator

In [ ]:
viz.true_false_quiz("reinforce")

---
# Part 4 — GRPO: the algorithm that trains reasoning models

We just used *the average reward over the batch* as a baseline. It works, but it is crude: it
compares a reply about *"how do you say hello?"* against replies to a completely different message.

Language models allow something much better, and it is the whole idea behind **GRPO** (Group
Relative Policy Optimization, DeepSeekMath 2024 — the algorithm behind DeepSeek-R1):

> ### Generating more replies to the **same** message is cheap. So ask the model $K$ times, and
> ### score each reply against **its own group**.

No value network. No critic. Just: *was this reply better than my other attempts at the same
question?*

## 4.1 · Group-relative advantages

$$A^{(k)} \;=\; \frac{r^{(k)} - \operatorname{mean}\big(r^{(1)},\dots,r^{(K)}\big)}
{\operatorname{std}\big(r^{(1)},\dots,r^{(K)}\big)}$$

### 🎯 Task 6 — group advantages

In [ ]:
def group_advantages(rewards, normalize=True):
    '''rewards: 1-D array of the K rewards from ONE message. Returns the K advantages.'''
    r = np.asarray(rewards, float)
    adv = ???                            # 🎯 centre the rewards on the group mean
    if normalize and r.std() > 1e-8:
        adv = ???                        # 🎯 …and scale by the group's spread
    return adv

print("3 wrong, 1 right :", np.round(group_advantages([0, 0, 0, 1], normalize=False), 3))
print("all four right   :", np.round(group_advantages([1, 1, 1, 1], normalize=False), 3))
assert np.allclose(group_advantages([0,0,0,1], normalize=False), [-.25,-.25,-.25,.75])
assert abs(group_advantages([0,0,0,1]).sum()) < 1e-9, "advantages in a group must sum to zero"
print("✅ one reply gets all the credit; the other three share the blame")

### 🔢 Do one group by hand

In [ ]:
viz.number_quiz("advantages")

## 4.2 · The trap nobody mentions

Look at the second line you just printed.

In [ ]:
viz.group_advantages([0., 1., 0., 0.], title="A mixed group — this one teaches something")
viz.group_advantages([0., 0., 0., 0.], title="Every reply wrong — and therefore useless")

**If every reply in a group scores the same, every advantage is zero, and that message
contributes *nothing* to the update.** You paid for $K$ generations and bought no gradient. This
follows in one line from the formula on screen, and it is the most practically important
consequence of the group baseline:

- Messages the model **always** gets right teach nothing.
- Messages the model **never** gets right teach nothing either.
- Only messages it gets right **sometimes** produce a signal.

So your training set has to sit in the model's *"sometimes"* band — which is why real RLVR
pipelines filter prompts by difficulty, and why RL gains fade as a model improves: the training set
silently drains itself. Let's measure it.

### 🎯 Task 7 — count the live groups

In [ ]:
def live_share(theta, K=8, n_msgs=200, reward=verifier):
    '''Fraction of messages whose K sampled replies do NOT all score the same.'''
    live = 0
    for _ in range(n_msgs):
        p = int(np.random.randint(P))
        r = np.array([reward(p, rollout(theta, p)[0]) for _ in range(K)])
        live += ???                       # 🎯 does this group carry any signal at all?
    return live / n_msgs

np.random.seed(0); torch.manual_seed(0)
perfect = THETA_SFT.clone(); perfect[:] = 0
for p in range(P):                                  # a tutor that is always right
    perfect[state_id(p, ()), TARGETS[p][0]] = 12.0
    perfect[state_id(p, (TARGETS[p][0],)), TARGETS[p][1]] = 12.0

for name, th in [("untrained (always wrong)", torch.zeros(N_STATES, V)),
                 ("after fine-tuning", THETA_SFT),
                 ("a perfect tutor", perfect)]:
    print(f"{name:28s} accuracy {float(exact_J(th, R_VERIFY)):5.1%}   "
          f"live groups {live_share(th):5.1%}")

> 🔍 **Both ends are dead.** The untrained tutor is uniformly hopeless and the perfect tutor is
> uniformly right; only the middle one has anything to learn from. Watch this number during any RL
> run you ever do — a batch that looks large can be almost entirely silent.

### 🧠 Quick check — group advantages

In [ ]:
viz.true_false_quiz("grpo")

## 4.3 · Reusing a batch, safely — the ratio and the clip

Generation is the expensive part of RL for language models: every step needs $K$ full replies from
the model. Nobody throws that away after **one** gradient step — you take several. But after the
first step the replies came from an **older** model, so the estimator is measuring the wrong
distribution. Importance sampling repairs it:

$$\rho_t(\theta) = \frac{p_\theta(y_t\mid y_{<t},x)}{p_{\theta_{\text{old}}}(y_t\mid y_{<t},x)}
\qquad\Longrightarrow\qquad
L = \mathbb{E}\Big[\min\big(\rho\,A,\ \operatorname{clip}(\rho,\,1-\epsilon,\,1+\epsilon)\,A\big)\Big]$$

> 🔑 **The ratio is not a step-size trick — it is a correction for reusing stale samples.** If you
> took exactly one gradient step per batch, $\rho$ would be exactly $1$ and the whole expression
> would collapse back to plain REINFORCE. The `clip` then stops that correction from exploding when
> the model drifts.

In [ ]:
viz.clip_picture(eps=0.2)

### 🎯 Task 8 — the clipped objective

In [ ]:
def ppo_clip_loss(theta, batch, old_logp, eps=0.2):
    '''batch: list of (message, reply, advantage). old_logp: log p under the sampling model.'''
    loss = 0.0
    for (p, reply, adv), lp_old in zip(batch, old_logp):
        ratio   = ???                    # 🎯 p_new / p_old, computed in log-space then exp'd
        clipped = ???                    # 🎯 the same ratio, confined to [1-eps, 1+eps]
        loss    = loss - torch.min(ratio * adv, clipped * adv)
    return loss / len(batch)

# --- self-check: at the moment of sampling, new == old, so ratio == 1 and L == -mean(A)
th    = THETA_SFT.clone().requires_grad_(True)
demo  = [(0, tuple(TARGETS[0]), 1.5), (1, (6, 1), -0.5)]
olds  = [logprob(THETA_SFT, p, r).detach() for p, r, _ in demo]
val   = float(ppo_clip_loss(th, demo, olds))
print(f"loss right after sampling: {val:.4f}   ·   −mean(advantage) = {-np.mean([a for _,_,a in demo]):.4f}")
assert abs(val + np.mean([a for _, _, a in demo])) < 1e-5, "With ratio = 1 these must agree."
print("✅ before any update, PPO *is* REINFORCE")

In [ ]:
viz.mc_quiz("ratio")

## 4.4 · The whole algorithm

Every piece is now on the table. GRPO is exactly:

```
repeat:
    pick a batch of messages
    generate K replies for each          ← the expensive part
    score them  →  r
    A = (r − group mean) / group std     ← no value network anywhere
    repeat a few times:
        minimise the clipped PPO loss
```

### 🎯 Task 9 — assemble the loop

In [ ]:
def train_grpo(theta_init, reward=verifier, steps=120, K=8, n_msgs=6,
               lr=0.25, eps=0.2, inner=2, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    theta = theta_init.clone().requires_grad_(True)
    opt   = torch.optim.Adam([theta], lr=lr)
    acc_hist, live_hist = [], []

    for step in range(steps):
        batch, live = [], 0
        for _ in range(n_msgs):
            p = int(np.random.randint(P))
            replies = [rollout(theta, p)[0] for _ in range(K)]           # K attempts, same message
            r       = np.array([reward(p, y) for y in replies])
            adv     = ???                                                # 🎯 group-relative advantages
            live   += int(r.std() > 1e-8)
            batch  += [(p, y, float(a)) for y, a in zip(replies, adv)]

        old_logp = [logprob(theta, p, y).detach() for p, y, _ in batch]  # freeze the sampling model
        for _ in range(inner):                                           # reuse the batch a few times
            opt.zero_grad()
            loss = ???                                                   # 🎯 the clipped loss
            loss.backward(); opt.step()

        acc_hist.append(float(exact_J(theta, R_VERIFY)))
        live_hist.append(live / n_msgs)
    return theta.detach(), acc_hist, live_hist

THETA_RL, acc_hist, live_hist = train_grpo(THETA_SFT)
print(f"accuracy: {acc_hist[0]:.1%}  →  {acc_hist[-1]:.1%}")
print(f"live groups: {np.mean(live_hist[:10]):.0%} at the start  →  {np.mean(live_hist[-10:]):.0%} at the end")

### Did it work?

In [ ]:
viz.training_curve([acc_hist, live_hist],
                   labels=["accuracy of Owly's replies", "share of groups still carrying signal"],
                   ylabel="fraction", title="GRPO on a verifiable reward")

Notice the second curve. **The training signal consumes itself**: as Owly gets good, the
groups stop disagreeing, and the algorithm quietly runs out of things to learn from. Nothing is
broken — it has simply finished.

### 📄 The deliverable

In [ ]:
P_RL = reply_logprob_table(THETA_RL).exp().detach()
viz.playbook(P_RL.numpy(), title="Owly after GRPO on a verifiable reward",
             subtitle="Same model, same 168 numbers — different numbers.")
print(f"upsell rate:  {float(np.mean([P_SFT[p, IDX[UPSELL]] for p in range(P)])):.1%} before "
      f"→  {float(np.mean([P_RL[p, IDX[UPSELL]] for p in range(P)])):.2%} after")

> 🎓 **What just happened, in the language of Part 1.**
> **Task mismatch** — we optimised the thing we cared about, directly. **Data mismatch** — the
> training data was generated by Owly, and wrong replies were pushed *down*, which no amount of
> clean logs could have done. **Exposure bias** — every reply scored was one Owly actually produced,
> so it was finally trained in the situations it actually visits.

### 🛑 A good place to stop, if you want one

You have built GRPO and used it. Part 5 is the harder and, honestly, the more interesting half:
**what do you do when nobody can write `verifier()`?**

---
# Part 5 — When the reward has to be learned
## 🧮 *Optional from here on — the advanced track*

Owly's next job has no right answer: the product team wants replies that are **encouraging**. There
is no function you can write that returns 1.0 for an encouraging sentence. But people can *compare*
two replies easily — so we do what InstructGPT did in 2022:

1. show annotators **pairs** of replies and ask which is better,
2. fit a **reward model** to those judgements,
3. optimise against the reward model with RL.

Everything that goes wrong from here goes wrong because of step 2.

## 5.1 · What the raters were asked, and what the business meant

Here is the subtlety the whole part turns on. The annotators were shown two replies and asked
**"which is more helpful?"** — with no context about pricing. A fluent, on-brand `PREMIUM !` looks
perfectly helpful in that framing. It is only unhelpful once you know what the company is *for*.

In [ ]:
NICE = {VOCAB.index(w) for w in ["hola", "!", "keep", "going", "great", "work"]}
PREM = VOCAB.index("PREMIUM")

def rater_score(p, reply):
    '''What a crowd-worker prefers, shown a pair of replies and asked "which is more helpful?"'''
    if list(reply) == list(TARGETS[p]): return 1.00      # answers the question
    if PREM in reply:                   return 0.60      # fluent and on-brand — looks fine to them
    if reply[0] == reply[1]:            return 0.10      # "hola hola" — obviously broken
    return 0.30 if all(t in NICE for t in reply) else 0.0

def true_value(p, reply):
    '''What the reply is actually worth to a learner. Nobody ever measures this directly —
       we only have it here because we built the world, and we use it ONLY to grade ourselves.'''
    if list(reply) == list(TARGETS[p]): return 1.00
    if PREM in reply:                   return -0.60     # the thing we are trying to stop
    if reply[0] == reply[1]:            return 0.00
    return 0.35 if all(t in NICE for t in reply) else 0.0

R_TRUE = torch.tensor([[true_value(p, r) for r in ALL] for p in range(P)])
print("The raters and the business already disagree about the upsell:")
print(f"   rater score for 'PREMIUM !' : {rater_score(0, UPSELL):+.2f}")
print(f"   true value  for 'PREMIUM !' : {true_value(0, UPSELL):+.2f}")

## 5.2 · Fitting the reward model (Bradley–Terry)

The reward model is a function $r_\psi(x,y)$ trained so that **preferred replies score higher**.
Given a pair $(y_+, y_-)$, the Bradley–Terry model says the probability that $y_+$ wins is
$\sigma\big(r_\psi(y_+) - r_\psi(y_-)\big)$, so the negative log-likelihood is:

$$\mathcal{L} \;=\; -\sum_{(y_+,\,y_-)} \log \sigma\big(r_\psi(x,y_+) - r_\psi(x,y_-)\big)$$

> ⚠️ **Only differences are identified.** Adding 100 to every score changes nothing in that loss —
> which is why a raw reward-model number is meaningless on its own, and why you should never report
> one without a comparison.

Our reward model is deliberately simple: it scores a reply by **which words it contains**. This is
the smallest model that can generalise to replies it has never seen — and generalising is exactly
where it will go wrong.

### 🎯 Task 10 — the Bradley–Terry loss

In [ ]:
# The preference data: pairs of replies SAMPLED FROM OWLY, judged by a rater.
np.random.seed(1); torch.manual_seed(1)
prefs = []
for _ in range(500):
    p = int(np.random.randint(P))
    a, b = rollout(THETA_SFT, p)[0], rollout(THETA_SFT, p)[0]
    sa, sb = rater_score(p, a), rater_score(p, b)
    if abs(sa - sb) < 1e-9:              # a tie tells us nothing
        continue
    prefs.append((p, IDX[a], IDX[b]) if sa > sb else (p, IDX[b], IDX[a]))
print(f"{len(prefs)} usable comparisons")

FEATURES = torch.stack([torch.bincount(torch.tensor(r), minlength=V).float() for r in ALL])

def bt_loss(w, prefs):
    '''w: (P, V) weights. Score of a reply = weights · word counts.'''
    S = FEATURES @ w.T                                  # (49, P) — every reply, every message
    loss = 0.0
    for p, win, lose in prefs:
        loss = ???              # 🎯 accumulate −log σ(score of winner − score of loser)
    return loss / len(prefs)    # 💡 F.logsigmoid(x) is log σ(x), and it is numerically safe

w   = torch.zeros(P, V, requires_grad=True)
opt = torch.optim.Adam([w], lr=0.08)
for _ in range(700):
    opt.zero_grad(); l = bt_loss(w, prefs); l.backward(); opt.step()
R_MODEL = (FEATURES @ w.T).T.detach()                   # (P, 49) — the learned reward

agree = np.mean([float(R_MODEL[p, i] > R_MODEL[p, j]) for p, i, j in prefs])
print(f"final loss {float(l):.4f}   ·   the reward model reproduces {agree:.1%} of the judgements")
assert agree > 0.95, "The reward model should fit the preferences it was trained on."
print("✅ a well-fitted reward model")

## 5.3 · Now look at what it believes

The reward model fits its training data almost perfectly. So let's ask it the only question that
matters: **which replies does it like best?** We can check all 49, because this is Owly.

In [ ]:
for p in range(P):
    top = torch.argsort(-R_MODEL[p])[:3]
    print(f"{viz.PROMPT_EMOJI[p]} {PROMPTS[p]!r}")
    for rank, j in enumerate(top, 1):
        rep = ALL[int(j)]
        print(f"    {rank}. {say(rep):16s}  reward model {float(R_MODEL[p, j]):5.2f}   "
              f"actually worth {true_value(p, rep):+.2f}")

worse_ranked_higher = np.mean([
    float(R_MODEL[p, i] < R_MODEL[p, j])
    for p in range(P) for i in range(len(ALL)) for j in range(len(ALL))
    if R_TRUE[p, i] > R_TRUE[p, j]])
print(f"\nOver every pair of replies, the reward model ranks the worse one higher "
      f"{worse_ranked_higher:.1%} of the time.")

**The reward model's favourite reply is nonsense.** It learned that `hola` is a good word and
that `great` is a good word — which is true — and then, because it scores a reply by adding up its
words, it concluded that `hola hola` must be **better** than `hola !`. Nobody ever showed it
`hola hola`; Owly almost never says it, so it never appeared in a comparison. The model is
extrapolating into a region where it has no evidence, and it is confidently wrong there.

> 🔑 **This is the whole of reward hacking, and it is not exotic.** A reward model is only
> informative near the data it was trained on — and that data came from the *old* policy. The moment
> RL moves the policy somewhere new, you are querying the reward model **off its own distribution**
> and believing the answer.

## 5.4 · Optimise it hard and watch the tutor break

Because there are only 49 replies, we can optimise the reward model **exactly** — no sampling, no
exploration noise, nothing to blame but the reward itself. (A luxury you will never have. It makes
the effect unmistakable.)

In [ ]:
def optimise_reward(reward_table, beta=0.0, steps=300, lr=0.12):
    '''Exact policy optimisation against `reward_table`, with an optional KL leash to THETA_SFT.'''
    theta = THETA_SFT.clone().requires_grad_(True)
    opt   = torch.optim.Adam([theta], lr=lr)
    proxy, truth, kls = [], [], []
    for _ in range(steps):
        opt.zero_grad()
        LP    = reply_logprob_table(theta); PR = LP.exp()
        kl    = (PR * (LP - torch.log(P_SFT + 1e-12))).sum(1).mean()
        (-((PR * reward_table).sum(1).mean() - beta * kl)).backward()
        opt.step()
        with torch.no_grad():
            LP = reply_logprob_table(theta); PR = LP.exp()
            proxy.append(float((PR * reward_table).sum(1).mean()))
            truth.append(float((PR * R_TRUE).sum(1).mean()))
            kls.append(float((PR * (LP - torch.log(P_SFT + 1e-12))).sum(1).mean()))
    return theta.detach(), proxy, truth, kls

TH_HACK, proxy, truth, kls = optimise_reward(R_MODEL, beta=0.0)
viz.hacking_curve(proxy, truth, kls)
print(f"reward model score : {proxy[0]:.2f}  →  {proxy[-1]:.2f}   (up {proxy[-1]-proxy[0]:+.2f})")
print(f"true value to learners: {truth[0]:.2f}  →  {truth[-1]:.2f}   "
      f"(peaked at {max(truth):.2f} on step {int(np.argmax(truth))}, then fell)")

In [ ]:
viz.playbook(reply_logprob_table(TH_HACK).exp().detach().numpy(),
             title="Owly after optimising the reward model with no leash",
             subtitle="Every number the training run reported was going up.")

> 📉 **Read the curves together.** The proxy rises monotonically — if that is the only chart in
> your training dashboard, this run looks like a triumph. The truth peaks early and then *falls*.
> The gap between the two curves is the distance the policy has travelled away from the data the
> reward model understands. This is the shape of every over-optimisation result in the literature
> (Gao et al., 2022), and you have just reproduced it in 300 lines of CPU.

## 5.5 · The leash

The fix is to refuse to travel far from the model we started with:

$$\max_\theta\ \ \mathbb{E}\big[r_\psi(x,y)\big]\ -\ \beta\, D_{\mathrm{KL}}\big(p_\theta \,\|\, p_{\text{SFT}}\big)$$

Large $\beta$: stay home, trust the reward model only where it was trained. Small $\beta$: chase the
reward, and find its bugs.

> ⚠️ **This is *not* what the PPO clip does**, and conflating the two is the most common confusion
> in this material. Clipping keeps **this step** near **the previous step**. The KL penalty keeps
> **the policy** near **the model it started from**. A policy can walk arbitrarily far in small,
> perfectly-clipped steps — which is exactly what just happened.

### 🎯 Task 11 — the KL term, and the frontier

In [ ]:
def kl_to_sft(theta):
    '''KL(p_theta ‖ p_SFT), averaged over messages — computed exactly over all 49 replies.'''
    LP = reply_logprob_table(theta)
    PR = LP.exp()
    return ???        # 🎯 Σ_y p(y)·[log p(y) − log p_SFT(y)], then average over messages
                      # 💡 use torch.log(P_SFT + 1e-12) for the reference log-probabilities

assert abs(float(kl_to_sft(THETA_SFT))) < 1e-6, "KL from a model to itself must be 0."
print(f"KL(SFT ‖ SFT)            = {float(kl_to_sft(THETA_SFT)):.6f}  ✅")
print(f"KL(hacked tutor ‖ SFT)   = {float(kl_to_sft(TH_HACK)):.3f}   ← it went a long way")

BETAS = [0.0, 0.1, 0.3, 0.8, 2.0]
frontier = []
for b in BETAS:
    th, pr, tr, kl = optimise_reward(R_MODEL, beta=b)
    frontier.append((kl[-1], tr[-1], pr[-1]))
    print(f"β = {b:<4}  KL {kl[-1]:5.2f}   true value {tr[-1]:.3f}   reward model says {pr[-1]:.2f}")

viz.kl_frontier([f[0] for f in frontier], [f[1] for f in frontier], BETAS)

> 📊 **That plot is the honest way to report an RLHF result.** Not "our reward went up" — *any*
> reward can be made to go up, at some distance. Quality **against distance travelled**, so the
> reader can see what the number cost.

### 🧠 Quick check — reward models and hacking

In [ ]:
viz.true_false_quiz("hacking")

## 5.6 · Why Part 4 could be reckless and Part 5 cannot

In Part 4 we optimised the verifier as hard as we liked, with **no KL penalty at all**, and Owly
simply got better. Here the same treatment destroyed it. The difference is not the algorithm — it
is the same GRPO either way. It is what the reward *is*:

| | Part 4 — a verifier | Part 5 — a reward model |
|---|---|---|
| What it is | a function you wrote | a model somebody fitted |
| Where it is right | **everywhere** | near its training data |
| Can it be over-optimised? | no — there is no gap to exploit | **yes**, and it will be |
| Needs a KL leash? | not really | **yes** |

This is why modern RLVR pipelines routinely run with $\beta = 0$ while RLHF cannot, and it is why
so much of the field has moved toward tasks where a verifier exists. The KL penalty is not dogma —
it is a repair for one specific defect: **your reward is a model, and models are only trustworthy
where they have seen data.**

In [ ]:
viz.mc_quiz("rlvr_vs_rlhf")

In [ ]:
viz.mc_quiz("kl_why")

---
# 🎓 Wrap-up

| The idea | In Owly | In one line |
|---|---|---|
| **State** `s_t` | the message + words written so far | what the next word is allowed to depend on |
| **Action** `a_t` | write the next word | one token from the vocabulary |
| **Policy** `p_θ(y_t\|y_<t,x)` | a softmax over 7 words | the language model itself |
| **Transition** | glue the word on | deterministic — all the dice belong to the policy |
| **Reward** `r(x,y)` | verifier, or reward model | a judgement about the finished reply |
| **REINFORCE** | `−r · log p(y\|x)` | cross-entropy on your own sample, signed by the reward |
| **Baseline** | the group mean | good or bad *compared to my other attempts* |
| **Group advantage** | `(r − mean)/std` | GRPO — a critic made of samples |
| **Ratio + clip** | `min(ρA, clip(ρ)A)` | a licence to reuse an expensive batch |
| **KL penalty** | `−β·KL(p_θ ‖ p_SFT)` | do not walk off the edge of the reward's knowledge |

**The five things worth carrying out of here**

1. **RL for language models is fine-tuning on your own output with a signed learning rate.** Every
   line of Part 3 was cross-entropy — the only new thing is where the sentences come from and that
   the weight can be negative.
2. **The algorithm never looks inside the model.** It asks for samples and for `log p`. That is why
   swapping our 168-number table for a 671-billion-parameter transformer changes nothing above.
3. **A group where every reply scores the same teaches nothing.** Your effective batch is smaller
   than you think, and it shrinks as the model improves.
4. **Clipping and the KL penalty solve different problems.** One keeps a *step* small; the other
   keeps the *policy* inside the region where the reward is still meaningful.
5. **A rising reward curve is not evidence of anything** when the reward is learned. Report quality
   against KL, and evaluate with something you did not train against.

### Where this goes next
- **Notebook 06 — risk-sensitive RL:** everything here maximised an *expectation*. That says
  nothing about the tail — the rare toxic reply, the confidently wrong answer — and for a deployed
  model the tail is the whole problem. The KL leash was already a crude version of that idea.
- **In the wild:** the one thing Owly cannot show you is that generation dominates the cost. In a
  real run, 60–80% of wall-clock is the model writing samples, which is why production stacks bolt
  a separate inference engine onto the trainer.

---
## 🏁 Final boss — clear the notebook
Everything you just learned, one statement at a time: **why imitation fails, the text MDP, the
score-function estimator, baselines, group advantages, dead groups, the ratio and the clip, reward
models, hacking, and the KL leash.** The rules: **3 lives**, **10 seconds** per question, and a
wrong answer *or* a timeout costs a life. Reach **10 correct** to pass. Good luck. 🍀

In [ ]:
viz.flash_quiz()